# 氣象資料爬蟲＋LINE Bot＋GPT

In [ ]:
# 金鑰從 Colab 左側欄的 🔑 Secrets 讀取（不在 Colab 時改讀環境變數），不寫在程式裡
import os

def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

In [25]:
import requests

CWA_API_KEY = get_secret("CWA_API_KEY")
url = f'https://opendata.cwa.gov.tw/api/v1/rest/datastore/F-C0032-001?Authorization={CWA_API_KEY}&limit=100&offset=0&format=JSON&locationName=%E8%87%BA%E5%8C%97%E5%B8%82&elementName=&sort=time'
data = requests.get(url)
data_json = data.json()

locations = data_json['records']['location']  # Extracting locations

for location in locations:  # Loop through each location
    location_name = location['locationName']
    print("Location:", location_name)

    weather_elements = location['weatherElement']  # Extracting weather elements for each location

    for weather_element in weather_elements:  # Loop through each weather element
        element_name = weather_element['elementName']
        print("Weather Element:", element_name)

        for time_info in weather_element['time']:  # Loop through each time info
            start_time = time_info['startTime']
            # end_time = time_info['endTime']
            parameter = time_info['parameter']
            parameter_name = parameter['parameterName']
            print("Start Time:", start_time.split()[0])
            # print("End Time:", end_time)
            print("Parameter:", parameter_name)
            print()


Location: 臺北市
Weather Element: Wx
Start Time: 2024-03-16
Parameter: 多雲短暫雨

Start Time: 2024-03-17
Parameter: 多雲短暫雨

Start Time: 2024-03-17
Parameter: 陰短暫陣雨

Weather Element: PoP
Start Time: 2024-03-16
Parameter: 30

Start Time: 2024-03-17
Parameter: 30

Start Time: 2024-03-17
Parameter: 30

Weather Element: MinT
Start Time: 2024-03-16
Parameter: 18

Start Time: 2024-03-17
Parameter: 18

Start Time: 2024-03-17
Parameter: 18

Weather Element: CI
Start Time: 2024-03-16
Parameter: 稍有寒意至舒適

Start Time: 2024-03-17
Parameter: 稍有寒意至舒適

Start Time: 2024-03-17
Parameter: 稍有寒意至舒適

Weather Element: MaxT
Start Time: 2024-03-16
Parameter: 21

Start Time: 2024-03-17
Parameter: 27

Start Time: 2024-03-17
Parameter: 23



In [40]:
import requests

CWA_API_KEY = get_secret("CWA_API_KEY")
url = f'https://opendata.cwa.gov.tw/api/v1/rest/datastore/F-C0032-001?Authorization={CWA_API_KEY}&limit=100&offset=0&format=JSON&locationName=%E8%87%BA%E5%8C%97%E5%B8%82&elementName=&sort=time'
data = requests.get(url)
data_json = data.json()

locations = data_json['records']['location']  # Extracting locations

# Create a dictionary to store weather information for each date
weather_data_by_date = {}

for location in locations:  # Loop through each location
    location_name = location['locationName']

    weather_elements = location['weatherElement']  # Extracting weather elements for each location

    for weather_element in weather_elements:  # Loop through each weather element
        element_name = weather_element['elementName']

        for time_info in weather_element['time']:  # Loop through each time info
            start_time = time_info['startTime']
            end_time = time_info['endTime']
            parameter = time_info['parameter']
            parameter_name = parameter['parameterName']

            # Check if the date already exists in the dictionary
            if start_time in weather_data_by_date:
                # If date exists, append weather information to existing list
                weather_data_by_date[start_time].append((location_name, element_name, parameter_name))
            else:
                # If date does not exist, create a new list with weather information
                weather_data_by_date[start_time] = [(location_name, element_name, parameter_name)]

print(location_name)
print()

# Print the weather information for each date
for date, weather_info in weather_data_by_date.items():
    print("日期:", date)
    weather_list = {'天氣概況': [], '降雨機率': [], '天氣溫度': []}  # Create a dictionary to store weather information
    for info in weather_info:
        element_name, parameter_name = info[1], info[2]
        if element_name == "Wx" or element_name == "CI":
            weather_list['天氣概況'].append(parameter_name)
        elif element_name == "PoP":
            weather_list['降雨機率'].append(parameter_name + "%")
        elif element_name == "MinT" or element_name == "MaxT":
            weather_list['天氣溫度'].append(parameter_name)
    # Print the weather information for each category
    for category, values in weather_list.items():
        print(f"{category}:", ', '.join(values))
    print()


臺北市

日期: 2024-03-16 18:00:00
天氣概況: 多雲短暫雨, 稍有寒意至舒適
降雨機率: 30%
天氣溫度: 18, 21

日期: 2024-03-17 06:00:00
天氣概況: 多雲短暫雨, 稍有寒意至舒適
降雨機率: 30%
天氣溫度: 18, 27

日期: 2024-03-17 18:00:00
天氣概況: 陰短暫陣雨, 稍有寒意至舒適
降雨機率: 30%
天氣溫度: 18, 23



LineBot

In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent
)

app = Flask(__name__)

configuration = Configuration(access_token=get_secret("LINE_CHANNEL_ACCESS_TOKEN"))
handler = WebhookHandler(get_secret("LINE_CHANNEL_SECRET"))


@app.route("/callback", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        line_bot_api.reply_message_with_http_info(
            ReplyMessageRequest(
                reply_token=event.reply_token,
                messages=[TextMessage(text=event.message.text)]
            )
        )

if __name__ == "__main__":
    app.run()

In [3]:
!pip install --upgrade openai
!pip show openai
!pip install flask
!pip install line-bot-sdk

In [6]:
from flask import Flask, request, abort
from openai import OpenAI
import os

from linebot.v3 import (
  WebhookHandler
)
from linebot.v3.exceptions import (
  InvalidSignatureError
)
from linebot.v3.messaging import (
  Configuration,
  ApiClient,
  MessagingApi,
  ReplyMessageRequest,
  TextMessage
)
from linebot.v3.webhooks import (
  MessageEvent,
  TextMessageContent
)

app = Flask(__name__)

# 使用環境變數的名稱來獲取值
line_access_token = get_secret("LINE_CHANNEL_ACCESS_TOKEN")
# line_channel_secret = get_secret("LINE_CHANNEL_SECRET")
openai_api_key = get_secret("OPENAI_API_KEY")

configuration = Configuration(access_token= line_access_token)
handler = WebhookHandler(get_secret("LINE_CHANNEL_SECRET"))

@app.route("/callback", methods=['POST'])
def callback():
  # get X-Line-Signature header value
  signature = request.headers['X-Line-Signature']

  # get request body as text
  body = request.get_data(as_text=True)
  app.logger.info("Request body: " + body)

  # handle webhook body
  try:
      handler.handle(body, signature)
  except InvalidSignatureError:
      app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
      abort(400)

  return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
  # 使用 OpenAI API 生成回覆
  client = OpenAI(api_key= openai_api_key)
  response = client.chat.completions.create(
      model="gpt-3.5-turbo",
      messages=[
          {
              "role": "system",
              "content": "You are a helpful assistant."
          },
          {
              "role": "user",
              "content": event.message.text
          }
      ],
      temperature=0.5,
      max_tokens=64,
      top_p=1
  )

  reply_msg = response.choices[0].message["content"]

  with ApiClient(configuration) as api_client:
      line_bot_api = MessagingApi(api_client)
      line_bot_api.reply_message_with_http_info(
          ReplyMessageRequest(
              reply_token=event.reply_token,
              messages=[TextMessage(text=reply_msg)]
          )
      )

if __name__ == "__main__":
  app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


In [ ]:
from flask import Flask, request, abort
from openai import OpenAI
import os

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent
)

app = Flask(__name__)

# 使用環境變數的名稱來獲取值
# line_access_token = get_secret("LINE_CHANNEL_ACCESS_TOKEN")
# line_channel_secret = get_secret("LINE_CHANNEL_SECRET")
# openai_api_key = get_secret("OPENAI_API_KEY")

configuration = Configuration(access_token= get_secret("LINE_CHANNEL_ACCESS_TOKEN"))
handler = WebhookHandler(get_secret("LINE_CHANNEL_SECRET"))

@app.route("/callback", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    # 使用 OpenAI API 生成回覆
    client = OpenAI(api_key= get_secret("OPENAI_API_KEY"))
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant."
            },
            {
                "role": "user",
                "content": event.message.text
            }
        ],
        temperature=0.5,
        max_tokens=64,
        top_p=1
    )

    reply_msg = response.choices[0].message["content"]

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        line_bot_api.reply_message_with_http_info(
            ReplyMessageRequest(
                reply_token=event.reply_token,
                messages=[TextMessage(text=reply_msg)]
            )
        )

if __name__ == "__main__":
    app.run()